In [4]:
# Task: Build a Simple RAG System
# Generate Embeddings: Use an LLM (e.g., BERT) to generate embeddings for a set of documents.
# Store Embeddings: Insert the embeddings into a vector database (e.g., Pinecone or FAISS).
# Retrieve Documents: Perform a similarity search to retrieve relevant documents for a given query.
# Generate Response: Use the retrieved documents to generate a response using an LLM (e.g., GPT).

# Example Code:
# Step 0: Install necessary libraries
!pip install pinecone-client[grpc] transformers torch

# Step 1: Import dependencies
from transformers import BertTokenizer, BertModel
import torch
import numpy as np
from pinecone import Pinecone, ServerlessSpec
from time import sleep

# Step 2: Load BERT model and tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')

# Step 3: Embedding generator function
def generate_embedding(text):
    inputs = tokenizer(text, return_tensors='pt', truncation=True, padding=True)
    with torch.no_grad():
        outputs = model(**inputs)
    return outputs.last_hidden_state.mean(dim=1).squeeze().numpy()

# Step 4: Set Pinecone API key
api_key = "pcsk_6bcmby_GecYJfVr9F6naaQXnjZh3DDWSqibGRMsPgNCCuPUqr1fW3NF1obtTTgz9cZgz8P"
index_name = "document-embeddings"
dimension = 768

# Step 5: Try different cloud/region combinations to find one that works
possible_regions = [
    {"cloud": "gcp", "region": "starter"},
    {"cloud": "aws", "region": "us-east-1"},
    {"cloud": "gcp", "region": "us-west1"},
    {"cloud": "aws", "region": "us-west-2"},
]

working_spec = None
pc = Pinecone(api_key=api_key)

for spec in possible_regions:
    print(f"🔍 Trying cloud='{spec['cloud']}', region='{spec['region']}'...")
    try:
        if index_name not in pc.list_indexes().names():
            pc.create_index(
                name=index_name,
                dimension=dimension,
                metric="cosine",
                spec=ServerlessSpec(
                    cloud=spec['cloud'],
                    region=spec['region']
                )
            )
            print(f"✅ SUCCESS: Index created using {spec['cloud']} / {spec['region']}")
        else:
            print(f"✔️ Index already exists using {spec['cloud']} / {spec['region']}")

        working_spec = spec
        break  # Stop after finding a working one
    except Exception as e:
        print(f"❌ Failed with {spec['cloud']} / {spec['region']} → {type(e).__name__}: {e}")
        sleep(1)

if not working_spec:
    raise RuntimeError("❗ No working cloud/region found for your Pinecone project.")

# Step 6: Connect to the index
index = pc.Index(index_name)

# Step 7: Prepare your documents
documents = [
    "Document 1 text about climate change and global warming.",
    "Document 2 text about artificial intelligence and machine learning.",
    "Document 3 text about history of the Roman Empire."
]

# Step 8: Generate embeddings for documents
embeddings = [generate_embedding(doc) for doc in documents]

# Step 9: Upload vectors to Pinecone index
index.upsert(vectors=[
    (f"doc{i}", embedding.tolist()) for i, embedding in enumerate(embeddings)
])

# Step 10: Perform similarity search with a query
query = "Tell me about AI and technology."
query_embedding = generate_embedding(query)

results = index.query(vector=query_embedding.tolist(), top_k=2, include_metadata=False)

# Step 11: Retrieve relevant documents
retrieved_docs = []
for match in results['matches']:
    doc_id = int(match['id'][-1])  # Assumes IDs are "doc0", "doc1", etc.
    retrieved_docs.append(documents[doc_id])

# Step 12: Print the results
print("\n📄 Retrieved Documents for the Query:")
for doc in retrieved_docs:
    print("-", doc)


🔍 Trying cloud='gcp', region='starter'...
❌ Failed with gcp / starter → NotFoundException: (404)
Reason: Not Found
HTTP response headers: HTTPHeaderDict({'content-type': 'text/plain; charset=utf-8', 'access-control-allow-origin': '*', 'vary': 'origin,access-control-request-method,access-control-request-headers', 'access-control-expose-headers': '*', 'x-pinecone-api-version': '2025-01', 'X-Cloud-Trace-Context': 'c34c237b331ec74f0361303f97a5dade', 'Date': 'Thu, 13 Mar 2025 11:19:44 GMT', 'Server': 'Google Frontend', 'Content-Length': '101', 'Via': '1.1 google', 'Alt-Svc': 'h3=":443"; ma=2592000,h3-29=":443"; ma=2592000'})
HTTP response body: {"error":{"code":"NOT_FOUND","message":"Resource cloud: gcp region: starter not found"},"status":404}

🔍 Trying cloud='aws', region='us-east-1'...
✅ SUCCESS: Index created using aws / us-east-1

📄 Retrieved Documents for the Query:
